# Original YOLO12-Small pretrained — RDD2022 China

Baseline YOLO12s tanpa modifikasi arsitektur. Notebook ini memasang Ultralytics resmi, melakukan `train → val → test`, dan menyimpan seluruh hasil ke ZIP. Gunakan hyperparameter serta split data yang sama dengan eksperimen modifikasi agar perbandingan mAP adil.

In [ ]:
# 1. Install Ultralytics resmi. Aktifkan GPU dan Internet pada Kaggle.
import json
import platform
import subprocess
import sys
import zipfile
from pathlib import Path

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'ultralytics'], check=True)

import torch
import ultralytics
from ultralytics import YOLO

WORKDIR = Path('/kaggle/working')
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

def log_section(title: str) -> None:
    print(f'\n{"=" * 88}\n{title}\n{"=" * 88}')

log_section('ENVIRONMENT')
print(f'Python      : {platform.python_version()}')
print(f'PyTorch     : {torch.__version__}')
print(f'Ultralytics : {ultralytics.__version__}')
print(f'Device      : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')


In [ ]:
# 2. Dataset RDD2022 China dan hyperparameter baseline.
DATA_ROOT = Path('/kaggle/input/datasets/danialalfayyadh/ch-rdd2022/datasets-china-split')
DATA_YAML = WORKDIR / 'ch_rdd2022.yaml'

EPOCHS, IMGSZ, BATCH, NBS = 160, 640, 16, 64
OPTIMIZER, LR0, MOMENTUM, WEIGHT_DECAY = 'SGD', 0.01, 0.937, 0.0005
PATIENCE, WORKERS, SEED = 0, 2, 42
EXPERIMENT_NAME = 'yolo12s_original_pretrained_ch_rdd2022'
RUNS_DIR = WORKDIR / 'runs'

DATA_YAML.write_text(f'''path: {DATA_ROOT}
train: train/images
val: val/images
test: test/images

nc: 5
names:
  0: D00
  1: D10
  2: D20
  3: D40
  4: Repair
''', encoding='utf-8')
assert DATA_ROOT.exists(), f'Dataset path tidak ditemukan: {DATA_ROOT}'
print(DATA_YAML.read_text(encoding='utf-8'))
print(f'epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, nbs={NBS}, optimizer={OPTIMIZER}, lr0={LR0}, seed={SEED}')


In [ ]:
# 3. Training original pretrained YOLO12s. yolo12s.pt akan diunduh otomatis dari Ultralytics bila belum tersedia.
log_section('TRAINING — ORIGINAL PRETRAINED YOLO12S')
model = YOLO('yolo12s.pt')
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, nbs=NBS,
    device=DEVICE, workers=WORKERS,
    project=str(RUNS_DIR), name=EXPERIMENT_NAME, exist_ok=True,
    pretrained=True, optimizer=OPTIMIZER,
    lr0=LR0, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY,
    cos_lr=False, patience=PATIENCE, seed=SEED,
    plots=True, verbose=True,
)
RUN_DIR = Path(model.trainer.save_dir)
BEST_PT, LAST_PT = Path(model.trainer.best), Path(model.trainer.last)
print(f'Run directory: {RUN_DIR}')
print(f'Best weights : {BEST_PT}')


In [ ]:
# 4. Validasi best.pt pada validation split.
log_section('VALIDATION — BEST.PT')
best_model = YOLO(str(BEST_PT))
val_metrics = best_model.val(
    data=str(DATA_YAML), split='val', imgsz=IMGSZ, batch=BATCH, device=DEVICE,
    project=str(RUNS_DIR), name=f'{EXPERIMENT_NAME}_val', exist_ok=True, plots=True,
)

def metric_summary(metrics) -> dict:
    return {
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
        'map50': float(metrics.box.map50),
        'map50_95': float(metrics.box.map),
        'save_dir': str(metrics.save_dir),
    }

VAL_REPORT = metric_summary(val_metrics)
print(json.dumps(VAL_REPORT, indent=2))


In [ ]:
# 5. Uji best.pt pada test split, lalu kompres artefak training, validation, dan test ke ZIP.
log_section('TEST — BEST.PT')
test_label_dir = DATA_ROOT / 'test' / 'labels'
if test_label_dir.exists() and any(test_label_dir.glob('*.txt')):
    test_metrics = best_model.val(
        data=str(DATA_YAML), split='test', imgsz=IMGSZ, batch=BATCH, device=DEVICE,
        project=str(RUNS_DIR), name=f'{EXPERIMENT_NAME}_test', exist_ok=True, plots=True,
    )
    TEST_REPORT = metric_summary(test_metrics)
    TEST_OUTPUT_DIR = Path(test_metrics.save_dir)
else:
    predictions = best_model.predict(
        source=str(DATA_ROOT / 'test' / 'images'), imgsz=IMGSZ, device=DEVICE, conf=0.25,
        save=True, save_txt=True, project=str(RUNS_DIR),
        name=f'{EXPERIMENT_NAME}_test_predictions', exist_ok=True,
    )
    TEST_REPORT = {'status': 'test labels unavailable; prediction only'}
    TEST_OUTPUT_DIR = Path(predictions[0].save_dir) if predictions else RUNS_DIR

EVALUATION_REPORT = {'validation': VAL_REPORT, 'test': TEST_REPORT}
EVALUATION_JSON = WORKDIR / f'{EXPERIMENT_NAME}_evaluation_metrics.json'
EVALUATION_JSON.write_text(json.dumps(EVALUATION_REPORT, indent=2), encoding='utf-8')
RUN_CONFIG = WORKDIR / f'{EXPERIMENT_NAME}_config.json'
RUN_CONFIG.write_text(json.dumps({
    'model': 'yolo12s.pt', 'dataset_root': str(DATA_ROOT), 'data_yaml': str(DATA_YAML),
    'epochs': EPOCHS, 'imgsz': IMGSZ, 'batch': BATCH, 'nbs': NBS,
    'optimizer': OPTIMIZER, 'lr0': LR0, 'momentum': MOMENTUM,
    'weight_decay': WEIGHT_DECAY, 'seed': SEED,
    'best_checkpoint': str(BEST_PT), 'last_checkpoint': str(LAST_PT),
}, indent=2), encoding='utf-8')

ZIP_PATH = WORKDIR / f'{EXPERIMENT_NAME}_results.zip'
def add_to_zip(archive: zipfile.ZipFile, path: Path) -> int:
    if not path.exists():
        return 0
    files = [path] if path.is_file() else [item for item in path.rglob('*') if item.is_file()]
    for file_path in files:
        archive.write(file_path, file_path.relative_to(WORKDIR))
    return len(files)

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    count = sum(add_to_zip(archive, item) for item in (
        RUN_DIR, Path(val_metrics.save_dir), TEST_OUTPUT_DIR, DATA_YAML, RUN_CONFIG, EVALUATION_JSON,
    ))
print(json.dumps(EVALUATION_REPORT, indent=2))
print(f'ZIP created : {ZIP_PATH} ({count} files)')
from IPython.display import FileLink, display
display(FileLink(ZIP_PATH))
